# ModelGuard – AI Lifecycle Management & Governance Agent

## Notebook 02: MLflow Experiment Tracking and Model Management

## Notebook Overview

This notebook focuses on applying MLOps practices using MLflow to manage machine learning experiments and models.

After developing and evaluating the customer churn prediction models in Notebook 01, the next step is to organize and track the experiments. MLflow will be used to record model parameters, evaluation metrics, and trained model artifacts.

The selected XGBoost model will be registered for future use in model monitoring and deployment.

## Objectives

- Configure MLflow for experiment tracking.
- Create an experiment for customer churn prediction.
- Record model parameters and evaluation metrics.
- Save trained models as MLflow artifacts.
- Register the selected XGBoost model.
- Prepare the model for future lifecycle stages.

## Environment Setup

This section imports the required libraries and checks the installed versions before configuring MLflow.

In [1]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

import pandas as pd
import numpy as np
import sklearn
import xgboost

print("MLflow version:", mlflow.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgboost.__version__)

print("Environment setup completed successfully.")

MLflow version: 3.18.0
Scikit-learn version: 1.9.1
XGBoost version: 3.4.1
Environment setup completed successfully.


## Configure MLflow Tracking

MLflow Tracking is used to record machine learning experiments.
It stores parameters, evaluation metrics, and model artifacts
in one organized location.

In this project, a local SQLite database will be used to store
experiment information.

In [2]:
from pathlib import Path
import mlflow

# Configure local MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Create or select the experiment
experiment_name = "ModelGuard_Churn_Prediction"

mlflow.set_experiment(experiment_name)

# Get experiment information
experiment = mlflow.get_experiment_by_name(experiment_name)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Artifact Location:", experiment.artifact_location)

MLflow Tracking URI: sqlite:///mlflow.db
Experiment Name: ModelGuard_Churn_Prediction
Experiment ID: 1
Artifact Location: file:C:/Users/User/ModelGuard/mlruns/1


## Loading Exported Models and Results

This section loads the trained models and experiment results
exported from Notebook 01.

In [3]:
import json
import joblib
from pathlib import Path

# Define artifacts directory
PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

# Load experiment results
with open(ARTIFACTS_DIR / "model_results.json", "r", encoding="utf-8") as file:
    model_results = json.load(file)

# Load trained models
logistic_model = joblib.load(
    ARTIFACTS_DIR / "logistic_regression.joblib"
)

xgboost_model = joblib.load(
    ARTIFACTS_DIR / "final_xgboost_model.joblib"
)

print("Models and results loaded successfully.")

print("\nAvailable experiments:")
for experiment in model_results:
    print("-", experiment)

print("\nLoaded models:")
print("- Logistic Regression")
print("- XGBoost")

Models and results loaded successfully.

Available experiments:
- logistic_regression
- xgboost
- xgboost_undersampling

Loaded models:
- Logistic Regression
- XGBoost


## Inspect the structure of exported experiment results

In [4]:
for experiment_name, experiment_data in model_results.items():
    print(f"\n{'=' * 60}")
    print(f"Experiment: {experiment_name}")
    print(f"{'=' * 60}")

    for section_name, section_data in experiment_data.items():
        print(f"\n{section_name}:")

        if isinstance(section_data, dict):
            for key, value in section_data.items():
                if isinstance(value, dict):
                    print(f"  {key}:")
                    for metric, metric_value in value.items():
                        print(f"    {metric}: {metric_value}")
                elif isinstance(value, list):
                    print(f"  {key}: {len(value)} records")
                    if value:
                        print(f"    First record: {value[0]}")
                else:
                    print(f"  {key}: {value}")

        elif isinstance(section_data, list):
            print(f"  Number of records: {len(section_data)}")
            if section_data:
                print(f"  First record: {section_data[0]}")

        else:
            print(f"  {section_data}")


Experiment: logistic_regression

model_role:
  Baseline

cross_validation:
  Baseline:
    precision: 0.6529272596206235
    precision_std: 0.02617550991267593
    recall: 0.5431438127090301
    recall_std: 0.04110312105882869
    f1: 0.5923285102625757
    f1_std: 0.029850897224643135
    roc_auc: 0.8462007441910661
    roc_auc_std: 0.012576130140361557
    pr_auc: 0.6616561497949907
    pr_auc_std: 0.019353571740782758
  Balanced:
    precision: 0.5169334242384205
    precision_std: 0.018898965297489835
    recall: 0.8013377926421403
    recall_std: 0.037932968237479145
    f1: 0.6282570726199064
    f1_std: 0.023165361233898393
    roc_auc: 0.8459527189246974
    roc_auc_std: 0.012409919912800488
    pr_auc: 0.6599847261254812
    pr_auc_std: 0.0194756703770003

threshold_optimization:
  selected_threshold: 0.6

test_results:
  Number of records: 2
  First record: {'Threshold': 0.5, 'accuracy': 0.7381121362668559, 'Precision': 0.504302925989673, 'Recall': 0.7834224598930482, 'F1-sc

In [5]:
import mlflow
from pathlib import Path

# Set MLflow experiment
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ModelGuard_Churn_Prediction")

# Define artifacts directory
ARTIFACTS_DIR = Path.cwd() / "artifacts"


def log_cv_metrics(results, prefix):
    """Log cross-validation means and standard deviations."""

    for metric, value in results.items():
        if metric.endswith("_std"):
            metric_name = metric[:-4]
            mlflow.log_metric(
                f"{prefix}_{metric_name}_std",
                float(value)
            )
        elif isinstance(value, (int, float)):
            mlflow.log_metric(
                f"{prefix}_{metric}",
                float(value)
            )


def log_test_metrics(results, prefix):
    """Log test metrics while handling different result formats."""

    if isinstance(results, list):
        for record in results:
            threshold = record.get("Threshold", 0.5)
            threshold_label = str(threshold).replace(".", "_")

            for metric, value in record.items():
                if metric == "Threshold":
                    continue

                metric_name = metric.lower().replace("-", "_")

                if metric_name == "accurary":
                    metric_name = "accuracy"

                mlflow.log_metric(
                    f"{prefix}_threshold_{threshold_label}_{metric_name}",
                    float(value)
                )

    elif isinstance(results, dict):
        for metric, value in results.items():
            metric_name = metric.lower().replace("-", "_").replace(" ", "_")

            if isinstance(value, (int, float)):
                mlflow.log_metric(
                    f"{prefix}_{metric_name}",
                    float(value)
                )


# --------------------------------------------------
# Run 1: Logistic Regression
# --------------------------------------------------

lr_data = model_results["logistic_regression"]

with mlflow.start_run(run_name="Logistic Regression"):

    mlflow.set_tags({
        "model_type": "Logistic Regression",
        "role": "Baseline"
    })

    mlflow.log_param("class_weight_comparison", "Baseline vs Balanced")
    mlflow.log_param(
        "selected_threshold",
        lr_data["threshold_optimization"]["selected_threshold"]
    )

    # Cross-validation results
    for approach, metrics in lr_data["cross_validation"].items():
        log_cv_metrics(metrics, f"cv_{approach.lower()}")

    # Test results at both thresholds
    log_test_metrics(
        lr_data["test_results"],
        "test"
    )

    # Save model and experiment results as MLflow artifacts
    mlflow.log_artifact(
        str(ARTIFACTS_DIR / "logistic_regression.joblib")
    )

    mlflow.log_dict(
        lr_data,
        "experiment_results.json"
    )

    print("Logistic Regression run completed.")


# --------------------------------------------------
# Run 2: XGBoost
# --------------------------------------------------

xgb_data = model_results["xgboost"]

with mlflow.start_run(run_name="XGBoost"):

    mlflow.set_tags({
        "model_type": "XGBoost",
        "role": "Final Model"
    })

    # Cross-validation results
    log_cv_metrics(
        xgb_data["cross_validation"],
        "cv_initial"
    )

    # Hyperparameter tuning
    tuning = xgb_data["hyperparameter_tuning"]

    mlflow.log_metric(
        "tuning_best_cv_f1",
        float(tuning["best_cv_f1"])
    )

    mlflow.log_param(
        "selected_threshold",
        tuning["selected_threshold"]
    )

    for parameter, value in tuning["best_parameters"].items():
        mlflow.log_param(parameter, value)

    # Final test results
    log_test_metrics(
        xgb_data["test_results"],
        "test"
    )

    # Save final model and experiment results
    mlflow.log_artifact(
        str(ARTIFACTS_DIR / "final_xgboost_model.joblib")
    )

    mlflow.log_dict(
        xgb_data,
        "experiment_results.json"
    )

    print("XGBoost run completed.")


# --------------------------------------------------
# Run 3: XGBoost with Random Undersampling
# --------------------------------------------------

undersampling_data = model_results["xgboost_undersampling"]

with mlflow.start_run(run_name="XGBoost - Random Undersampling"):

    mlflow.set_tags({
        "model_type": "XGBoost",
        "imbalance_handling": "Random Undersampling",
        "role": "Experimental",
        "evaluation_status": "Cross-validation only"
    })

    # Cross-validation results
    for metric, values in undersampling_data["cross_validation"].items():

        metric_name = metric.lower().replace("-", "_").replace(" ", "_")

        mlflow.log_metric(
            f"cv_{metric_name}",
            float(values["Mean Score"])
        )

        mlflow.log_metric(
            f"cv_{metric_name}_std",
            float(values["Std"])
        )

    mlflow.log_dict(
        undersampling_data,
        "experiment_results.json"
    )

    print("XGBoost Undersampling run completed.")


print("\nAll three MLflow runs completed successfully.")

Logistic Regression run completed.
XGBoost run completed.
XGBoost Undersampling run completed.

All three MLflow runs completed successfully.
